# Second-order channels -- Figure 5

Magnetic injection current and shift current of the four materials (TNN, GGA) at $B = 0$ and at the enlarged Zeeman field $\mu_B B = 80$ meV, with the energy shift and the phase-space factor of the moment (no velocity shift), $\nu = 30$ meV. Each curve is the norm $\|T\| = (\sum_{abc} T_{abc}^2)^{1/2}$ of the $C_3$-averaged tensor. The kernels are in the engine (injection: velocity form, shift: projector form).

## Load engine

In [ ]:
%run engine.ipynb

## Resolution

Paper settings, with a quick setting in the comment. The cached results in `data/` have the resolution in their file names, so changing a line here starts a fresh run for that section.

In [ ]:
N_ch = 360         # square grid, same point density as the paper's 240 reciprocal grid; 120 for a quick look
n_ch = 1200        # photon energies; 300 for a quick look

## Injection and shift current

Photon energies from $0.9E_g$ to $1.4E_{\max}$. Under a minute per material at paper settings.

In [ ]:
channels = {}
for m in materials:
    fn = f"data/channels_{m}_N{N_ch}_n{n_ch}.npz"
    if not os.path.exists(fn):
        a = params(m)["a"]
        omega = np.linspace(0.9 * gap(m), 1.4 * E_max(m), n_ch)
        out = dict(omega=omega)
        for tag, B in (("B0", 0.0), ("Z", B80)):
            eta, sigma = eta_mag(m, "TNN", B, omega, 0.03, bz_grid(N_ch), terms="ED", shift=True)
            out["eta_" + tag] = np.abs(eta_SI(a)) * np.sqrt((eta**2).sum(axis=(1, 2, 3)))
            out["shift_" + tag] = shift_SI(a) * np.sqrt((sigma**2).sum(axis=(1, 2, 3)))
        np.savez(fn, **out)
    channels[m] = dict(np.load(fn))
    d = channels[m]
    print(f"{m}: largest ||eta|| at B = 0 is {d['eta_B0'].max():.1e}, at 80 meV {d['eta_Z'].max():.3f} A m V^-2 s^-1;"
          f" largest ||sigma_shift|| {d['shift_B0'].max():.1f} and {d['shift_Z'].max():.1f} nm uA V^-2")

In [ ]:
def trimmed(m):
    # cut the omega axis 15 points after the last point where any curve is above 0.3% of its maximum
    d = channels[m]
    big = (d["eta_Z"] > 0.003 * d["eta_Z"].max()) | (d["shift_Z"] > 0.003 * d["shift_Z"].max()) | (d["shift_B0"] > 0.003 * d["shift_B0"].max())
    k = np.where(big)[0][-1] + 15
    return [d[key][:k] for key in ("omega", "eta_B0", "eta_Z", "shift_B0", "shift_Z")]


fig = plt.figure(figsize=(TW, 5.6))
gs = fig.add_gridspec(3, 6, height_ratios=[1.35, 1, 1], hspace=0.55, wspace=1.1)
top = [fig.add_subplot(gs[0, 0:3]), fig.add_subplot(gs[0, 3:6])]
bottom = [[fig.add_subplot(gs[r, 2 * c:2 * c + 2]) for c in range(3)] for r in (1, 2)]
y_eta = r"$\|\eta^{a;(bc)}_{\mathrm{mag}}\|$ ($%s$)" % UNIT
y_shift = r"$\|\sigma^{a;(bc)}_{\mathrm{shift}}\|$ ($%s$)" % SUNIT

omega, eta0, etaZ, shift0, shiftZ = trimmed("MoS2")
for ax, y0, yZ, yl in ((top[0], eta0, etaZ, y_eta), (top[1], shift0, shiftZ, y_shift)):
    ax.plot(omega, y0, color=colors["MoS2"], lw=0.8, ls=":")
    ax.plot(omega, yZ, color=colors["MoS2"], lw=1.1)
    ax.set_ylabel(yl)
top[0].text(0.03, 0.9, names["MoS2"], transform=top[0].transAxes, va="top")
top[1].legend(handles=[Line2D([], [], color="k", lw=1.1, label=r"$\mu_BB=80$ meV"),
                       Line2D([], [], color="k", lw=0.8, ls=":", label=r"$B=0$")], loc="upper left")

for c, m in enumerate(materials[1:]):
    omega, eta0, etaZ, shift0, shiftZ = trimmed(m)
    for r, (y0, yZ) in enumerate(((eta0, etaZ), (shift0, shiftZ))):
        ax = bottom[r][c]
        ax.plot(omega, y0, color=colors[m], lw=0.7, ls=":")
        ax.plot(omega, yZ, color=colors[m], lw=1.0)
        ax.yaxis.set_major_locator(MaxNLocator(4))
        ax.xaxis.set_major_locator(MaxNLocator(4))
    bottom[0][c].text(0.05, 0.9, names[m], transform=bottom[0][c].transAxes, va="top")
bottom[0][0].set_ylabel(y_eta)
bottom[1][0].set_ylabel(y_shift)

for i, ax in enumerate(top + bottom[0] + bottom[1]):
    ax.set_xlabel(r"$\hbar\omega$ (eV)")
    ax.set_ylim(bottom=0)
    label(ax, "abcdefgh"[i])
save(fig, "nonlinear_channels_all")
plt.show()